# Fundamentos de Tensores, na Prática — Faixa Ouro (36-45 anos) e Sedentarismo

Objetivo: aprender o que é um tensor (escalar → vetor → matriz) resolvendo um problema real do GymSite — estimar o **potencial de alunos** por região, cruzando população da Faixa Ouro (35-45 anos) com a taxa de sedentarismo. No final, mostramos que o cálculo de oportunidade que já existe nos relatórios (o IOFO) é, estruturalmente, o mesmo cálculo que uma camada de rede neural faz.

**Fonte dos números — Passos 1 a 6:** `docs/arquitetura/faixa_ouro_3645_estrategia.md` (IBGE Censo 2022 + pesquisa setorial de sedentarismo, ver §1.1 e §2.2 do documento) — agregados regionais já publicados.

**Fonte dos números — Passo 7:** consulta ao vivo na tabela de produção `censo_setor_idade_sexo` (Supabase, projeto `epgedaiukjippepujuzc`), setor censitário → idade × sexo, Censo 2022, para o município de Fortaleza — números reais, não estimativa.

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

## Passo 1 — Tensor rank-0 (escalar)

Um escalar é um tensor de rank 0: um único número, sem eixos. É a forma mais simples — mas já carrega valor, base e fonte.

In [ ]:
# Faixa Ouro (35-45 anos) · Brasil · IBGE Censo 2022 (faixa_ouro_3645_estrategia.md §1.1)
populacao_faixa_ouro_brasil = 31_800_000
taxa_sedentarismo_nacional = 0.34  # pesquisa setorial, faixa 36-45

escalar = np.array(populacao_faixa_ouro_brasil)
print(escalar, "| rank:", escalar.ndim, "| shape:", escalar.shape)

## Passo 2 — Tensor rank-1 (vetor)

Um vetor tem um eixo: uma lista de números. Aqui, a % da população 35-45 anos e a taxa de sedentarismo estimada, por região (tabela §2.2 do documento).

In [ ]:
regioes = ["Sudeste", "Nordeste", "Sul", "Centro-Oeste", "Norte"]

# % da população 35-45 anos, por região (estimativa IBGE Censo 2022)
pct_pop_35_45 = np.array([0.18, 0.16, 0.17, 0.16, 0.14])

# taxa de sedentarismo estimada, ponto médio da faixa publicada (ex: Sudeste 32-35% -> 0.335)
taxa_sedentarismo_regional = np.array([0.335, 0.40, 0.30, 0.365, 0.425])

print("vetor % população :", pct_pop_35_45, "| shape:", pct_pop_35_45.shape)
print("vetor sedentarismo:", taxa_sedentarismo_regional, "| shape:", taxa_sedentarismo_regional.shape)

## Passo 3 — Tensor rank-2 (matriz)

Empilhando os dois vetores lado a lado, cada linha vira uma região e cada coluna uma variável — uma matriz (5 regiões × 2 variáveis).

In [ ]:
X = np.stack([pct_pop_35_45, taxa_sedentarismo_regional], axis=1)
print(X)
print("shape:", X.shape, "-> 5 regiões (linhas) x 2 variáveis (colunas)")

for regiao, linha in zip(regioes, X):
    print(f"{regiao:<13} pct_pop={linha[0]:.3f}  sedentarismo={linha[1]:.3f}")

## Passo 4 — Operações tensoriais: broadcasting e produto elemento-a-elemento

O documento define o **IOFO** (Índice de Oportunidade da Faixa Ouro, §5.1) como um produto de razões:

> IOFO = (Pop 36-45 / Pop Total) × (Sedentarismo Local / Sedentarismo Nacional) × (Renda Média / Renda Mínima)

Sem o fator de renda (não temos esse dado agregado por região aqui), a versão simplificada é um produto elemento-a-elemento entre a % de população e a razão de sedentarismo — exatamente uma operação de tensor com *broadcasting* (dividir o vetor inteiro por um escalar de uma vez).

In [ ]:
# broadcasting: o escalar taxa_sedentarismo_nacional é aplicado a cada elemento do vetor
razao_sedentarismo = taxa_sedentarismo_regional / taxa_sedentarismo_nacional
print("razão sedentarismo (local/nacional):", razao_sedentarismo)

# produto elemento-a-elemento (Hadamard) = IOFO simplificado por região
iofo_simplificado = pct_pop_35_45 * razao_sedentarismo
for regiao, valor in zip(regioes, iofo_simplificado):
    print(f"{regiao:<13} IOFO simplificado = {valor:.3f}")

## Passo 5 — Produto escalar (dot product): do IOFO ao neurônio

O IOFO multiplica os fatores. Uma rede neural, em vez de multiplicar, faz uma **soma ponderada**: cada variável recebe um peso, e a camada calcula `y = W·x + b` (produto escalar entre pesos e entradas, mais um viés). É o mesmo tipo de combinação — só troca produto por soma ponderada, e os pesos passam a ser ajustáveis (uma rede aprende esses pesos a partir de dados; aqui vamos escolhê-los manualmente para sentir o mecanismo).

In [ ]:
# pesos manuais: priorizamos sedentarismo (mercado menos disputado) sobre volume de população
pesos = np.array([0.4, 0.6])   # [peso pct_pop, peso razao_sedentarismo]
vies = -0.3

entradas = np.stack([pct_pop_35_45, razao_sedentarismo], axis=1)  # (5, 2)

score_linear = entradas @ pesos + vies   # produto escalar matriz-vetor: (5,2) @ (2,) -> (5,)
print("score linear (W·x + b) por região:", score_linear)

## Passo 6 — Ativação: transformando o score em decisão

Uma camada de rede neural não para no `W·x + b` — aplica uma **função de ativação** para transformar o score em algo interpretável (ex: uma probabilidade entre 0 e 1). Usamos a sigmoide.

**Atenção à escala**: os limiares do documento original para o IOFO (>1,2 acima da média; >1,5 excepcional) valem para a fórmula completa, com o fator de renda incluído. Como aqui removemos esse fator, a escala do índice simplificado mudou — por isso comparamos cada região contra a **média das 5 regiões**, não contra os limiares absolutos do documento. É o mesmo cuidado que vale para qualquer número: rótulo de proxy é obrigatório quando a fórmula é simplificada.

In [ ]:
def sigmoide(x):
    return 1 / (1 + np.exp(-x))

prioridade = sigmoide(score_linear)
media_iofo_simplificado = iofo_simplificado.mean()

for regiao, iofo, score, p in zip(regioes, iofo_simplificado, score_linear, prioridade):
    rotulo = "ACIMA DA MÉDIA" if iofo > media_iofo_simplificado else "ABAIXO DA MÉDIA"
    print(f"{regiao:<13} IOFO_simplificado={iofo:.3f} ({rotulo})   score_neuronio={score:.3f}   prioridade(sigmoide)={p:.3f}")

## Passo 7 — Dado real, ao vivo: matriz idade × sexo de Fortaleza

Os Passos 1-6 usaram agregados regionais já publicados. Agora trocamos por **dado real**, consultado ao vivo na tabela de produção `censo_setor_idade_sexo` (Supabase, projeto `epgedaiukjippepujuzc`), agregando os 4.280 setores censitários de Fortaleza (`id_municipio = '2304400'`).

**Carimbo do número:** 2.424.722 pessoas · agregado dos setores censitários de Fortaleza · Censo 2022 (`censo_setor_idade_sexo`, espelho BQ→Supabase) · consulta em 2026-08-26 — é um retrato (*snapshot*), não uma conexão ao vivo a cada execução deste notebook.

A tabela já vem organizada exatamente como uma **matriz (tensor rank-2)**: 4 faixas etárias (linhas) × 2 sexos (colunas) — a mesma segmentação que a pipeline de produção usa (`tools/data/censo2022_vcodes_idade_sexo.json`).

In [ ]:
# Fortaleza (id_municipio 2304400) · 4.280 setores censitários · Censo 2022
# censo_setor_idade_sexo (Supabase epgedaiukjippepujuzc) · consulta em 2026-08-26
faixas_etarias = ["15-24", "25-39 (core fitness)", "40-59", "60+"]
sexos = ["Homens", "Mulheres"]

fortaleza_pop_total = 2_424_722

# matriz real (4 faixas x 2 sexos)
fortaleza = np.array([
    [175_895, 176_933],   # 15-24
    [282_788, 316_494],   # 25-39
    [293_456, 363_678],   # 40-59
    [140_797, 220_858],   # 60+
])

print(fortaleza)
print("shape:", fortaleza.shape, "-> 4 faixas (linhas) x 2 sexos (colunas)")

# validação: a soma da matriz deve bater com o total do município
print("soma da matriz :", fortaleza.sum())
print("pop_total (BQ) :", fortaleza_pop_total)
print("diferença      :", fortaleza.sum() - fortaleza_pop_total, "(pessoas sem faixa etária classificada, ex: crianças 0-14)")

### Aplicando a taxa de sedentarismo na matriz real

A tabela `censo_setor_idade_sexo` não traz sedentarismo — isso vem da pesquisa setorial nacional (34%, usada nos Passos 1 e 4). Aqui reaproveitamos essa taxa nacional como aproximação para a faixa 25-39 de Fortaleza (a pesquisa original mede 36-45; **rótulo de proxy obrigatório**, é uma aproximação, não o dado exato dessa faixa nessa cidade). É `broadcasting` de novo: um escalar aplicado a toda a matriz.

In [ ]:
core_fitness = fortaleza[1]  # linha "25-39 (core fitness)": [homens, mulheres]
print("público 25-39 (Fortaleza):", core_fitness, "-> total:", core_fitness.sum())

# broadcasting: multiplica o vetor [homens, mulheres] pelo escalar de sedentarismo nacional (proxy)
sedentarios_core_fitness = core_fitness * taxa_sedentarismo_nacional
print("sedentários estimados (proxy 34%):", sedentarios_core_fitness.astype(int),
      "-> total:", int(sedentarios_core_fitness.sum()))

# reaproveitando o mesmo neurônio do Passo 5 (pesos, viés e sigmoide já definidos)
pct_pop_core_fitness_fortaleza = core_fitness.sum() / fortaleza_pop_total
razao_sedentarismo_fortaleza = taxa_sedentarismo_nacional / taxa_sedentarismo_nacional  # proxy nacional == nacional

entrada_fortaleza = np.array([pct_pop_core_fitness_fortaleza, razao_sedentarismo_fortaleza])
score_fortaleza = entrada_fortaleza @ pesos + vies
print(f"\nFortaleza: pct_pop_core_fitness={pct_pop_core_fitness_fortaleza:.3f}  "
      f"score_neuronio={score_fortaleza:.3f}  prioridade(sigmoide)={sigmoide(score_fortaleza):.3f}")

## Passo 8 — Treinando os pesos: gradiente descendente

Até aqui, `pesos = [0.4, 0.6]` e `vies = -0.3` foram **escolhidos à mão** — nós decidimos que sedentarismo pesa mais que volume de população. Isso funciona, mas não é "aprender": é uma regra de negócio disfarçada de rede neural.

Uma rede neural aprende os pesos a partir de **exemplos com resposta certa conhecida** (captação real observada por região). **Ainda não temos esse dado** — nenhuma campanha das 5 regiões foi executada e medida. Então, para provar o mecanismo sem inventar um resultado de negócio, fazemos o inverso do que normalmente se faz: geramos exemplos sintéticos a partir dos pesos que JÁ escolhemos no Passo 5, fingimos que não os conhecemos, e deixamos o gradiente descendente redescobri-los sozinho.

**O que provaria o mecanismo, de verdade**: não é o algoritmo devolver exatamente `[0.4, 0.6]` — sobre uma faixa limitada de dados, várias combinações de pesos podem produzir quase a mesma previsão final (as duas variáveis de entrada estão correlacionadas no intervalo gerado, então o modelo tem liberdade para compensar um peso com o outro). O que prova que o treino funcionou é a **previsão final** bater com a do Passo 5 — isso sim confirma que o algoritmo aprendeu a mesma função, mesmo chegando lá por um caminho de pesos diferente.

In [ ]:
# DADO SINTÉTICO — não é captação real observada, é gerado para provar o mecanismo de treino
np.random.seed(42)
n_amostras = 200

pesos_verdadeiros = np.array([0.4, 0.6])  # os mesmos do Passo 5 (o algoritmo "não sabe" disso)
vies_verdadeiro = -0.3

# faixas plausíveis: pct_pop entre 10% e 20%; razão de sedentarismo entre 0.7 e 1.4
x1 = np.random.uniform(0.10, 0.20, n_amostras)
x2 = np.random.uniform(0.70, 1.40, n_amostras)
X_treino = np.stack([x1, x2], axis=1)  # (200, 2)

ruido = np.random.normal(0, 0.05, n_amostras)
y_alvo = sigmoide(X_treino @ pesos_verdadeiros + vies_verdadeiro + ruido)  # (200,)

print("X_treino shape:", X_treino.shape, "| y_alvo shape:", y_alvo.shape)
print("primeiras 3 amostras:\n", X_treino[:3], "\ny_alvo:", y_alvo[:3])

### O laço de treino: forward, perda, gradiente, atualização

Cada rodada (época) faz 4 coisas:

1. **Forward**: calcula a previsão atual — `z = X·w + b`, `a = sigmoide(z)` (o mesmo cálculo do Passo 5-6, com pesos que começam aleatórios).
2. **Perda**: mede o quão errada está a previsão — erro quadrático médio entre `a` e `y_alvo`.
3. **Gradiente**: calcula, via regra da cadeia, o quanto cada peso contribuiu pro erro.
4. **Atualização**: move os pesos um pouco na direção que reduz o erro (`peso -= taxa_aprendizado × gradiente`).

In [ ]:
# inicialização aleatória — o treino começa "sem saber nada"
np.random.seed(0)
w = np.random.normal(0, 0.1, 2)
b = 0.0
taxa_aprendizado = 0.5
epocas = 2000
n = len(y_alvo)

historico_perda = []

for epoca in range(epocas):
    # 1. forward
    z = X_treino @ w + b
    a = sigmoide(z)

    # 2. perda (erro quadrático médio)
    perda = np.mean((a - y_alvo) ** 2)
    historico_perda.append(perda)

    # 3. gradiente (regra da cadeia: dL/da * da/dz * dz/dw)
    dL_da = 2 * (a - y_alvo) / n
    da_dz = a * (1 - a)
    delta = dL_da * da_dz               # (n,)
    grad_w = X_treino.T @ delta         # (2,)
    grad_b = delta.sum()

    # 4. atualização
    w -= taxa_aprendizado * grad_w
    b -= taxa_aprendizado * grad_b

    if epoca % 400 == 0:
        print(f"época {epoca:>4}  perda={perda:.5f}  w={w}  b={b:.3f}")

print(f"\npesos aprendidos : {w}   (verdadeiros: {pesos_verdadeiros})")
print(f"viés aprendido   : {b:.3f}   (verdadeiro: {vies_verdadeiro})")

### Validação: os pesos aprendidos reproduzem o Passo 5?

Aplicando `w` e `b` aprendidos nas 5 regiões do Passo 5: compare com `prioridade(sigmoide)` que já tínhamos calculado com os pesos escolhidos à mão. Repare que os pesos numéricos (`w`, `b`) provavelmente NÃO ficaram idênticos a `[0.4, 0.6]` e `-0.3` — o que importa é se a **previsão final** bate.

In [ ]:
prioridade_aprendida = sigmoide(entradas @ w + b)  # entradas = as 5 regiões do Passo 5

for regiao, p_manual, p_aprendida in zip(regioes, prioridade, prioridade_aprendida):
    print(f"{regiao:<13} prioridade(pesos à mão)={p_manual:.3f}   prioridade(pesos treinados)={p_aprendida:.3f}")

**Leitura do resultado**: rodando este notebook, os pesos aprendidos saíram bem diferentes dos verdadeiros (`w1≈0.17` vs `0.4`; `w2≈0.55` vs `0.6`), mas a prioridade final para cada região ficou a menos de 0,003 de diferença. O gradiente descendente encontrou **uma** função que explica os dados — não necessariamente **a mesma combinação de pesos** que gerou os dados. É um fenômeno real (não um bug do notebook): quando as variáveis de entrada têm baixa variação relativa entre si na amostra de treino, o modelo tem liberdade para redistribuir "crédito" entre os pesos sem piorar a previsão. Fica registrado porque é exatamente o tipo de armadilha que aparece quando se tenta interpretar os pesos de uma rede treinada como se fossem coeficientes de negócio — a previsão é confiável, o peso individual isolado nem sempre é.

## Onde isso te leva a seguir

1. ~~Trocar dado publicado por dado ao vivo~~ — **feito no Passo 7**, com a matriz real de Fortaleza. Próximo nível: repetir a consulta por setor censitário (não só o agregado do município) para localizar bairros específicos, como a pipeline de produção já faz.
2. ~~Pesos aprendidos, não escolhidos à mão~~ — **feito no Passo 8**, com dado sintético (ainda não existe captação real observada por região). O próximo nível real é trocar `y_alvo` por captação de campanha de fato medida assim que ela existir — aí o treino deixa de ser uma prova de conceito e vira uma ferramenta de negócio.
3. **Migrar para PyTorch**: depois de já enxergar os tensores e o gradiente "na mão" com NumPy, PyTorch introduz a mesma ideia com `torch.Tensor` e `autograd` (gradientes automáticos, sem escrever a regra da cadeia manualmente como no Passo 8) — faz sentido só depois de dominar os Passos 1-8 acima.